# clef quickstart

A tour of a running clef server from Python: classify, score, saved classifiers, many inputs, background jobs,
the OpenAI-compatible batch API, evaluation, the self-evolving router with its Review queue, and API keys.
Each section stands alone; run them in order the first time.

**Before you start**

1. Install the client in the notebook's environment: `pip install -e .` from the repo root (or `pip install clef-local`),
   plus `pip install openai` for section 6.
2. Start a server: `clef serve`. On the RX 7900 XTX the fastest settings are
   `CLEF_TUNABLEOP=1 CLEF_MASK_FREE=1 CLEF_FAST_HEAD=1 clef serve` (see section 9).
3. If the server is on another machine or has auth on, set `CLEF_URL` / `CLEF_API_KEY` below.

In [ ]:
import os

# Where the server is, and a key if auth is on (leave empty for a local server without keys).
os.environ.setdefault("CLEF_URL", "http://127.0.0.1:8910")
os.environ.setdefault("CLEF_API_KEY", "")
if not os.environ["CLEF_API_KEY"]:
    del os.environ["CLEF_API_KEY"]

from clef_client import ClefClient

clef = ClefClient()  # reads CLEF_URL / CLEF_API_KEY
health = clef.health()
print(health["status"], "| model:", health.get("model", {}).get("id"), "| backend:", health.get("backend"))

## 1. Classify one text

`classify` returns the best label, its confidence and a probability for every label. Labels can be a list, or a
dict of label → description (descriptions help when names are ambiguous).

In [ ]:
LABELS = {
    "billing": "Payments, invoices, refunds",
    "technical": "Bugs and outages",
    "account": "Login, profile, permissions",
}
r = clef.classify("Checkout is down, orders blocked since 03:00 UTC", LABELS)
print(r.label, round(r.confidence, 3))
print(r.scores)

### Multi-label and ordinal scores

`multi_label=True` returns every label above `threshold`. `score` places a text on an ordered scale.

In [ ]:
r = clef.classify("I was charged twice and the app crashes on login", LABELS, multi_label=True, threshold=0.5)
print("labels:", r.labels)

s = clef.score("Checkout is down, orders blocked", ["low", "medium", "high"], instructions="How urgent is this ticket?")
print("urgency:", s.level, "score", round(s.score, 2), "confidence", round(s.confidence, 2))

## 2. Saved classifiers

Save a label set once under a name and call it by name from anywhere (the console's Classifiers tab shows it too).

In [ ]:
triage = clef.classifier("notebook-triage")
triage.save(labels=LABELS, description="Support ticket routing (quickstart notebook)")
print(triage.classify("Please refund my last invoice").label)

## 3. Many inputs in one request

`classify_many` sends one request; clef batches the forwards on the GPU. Good up to a few hundred texts. For more,
use a job (next section).

In [ ]:
import csv
from pathlib import Path

repo = Path.cwd() if (Path.cwd() / "examples").exists() else Path.cwd().parent
tickets = [row["ticket"] for row in csv.DictReader(open(repo / "examples" / "tickets_labelled.csv"))]
for text, res in list(zip(tickets, clef.classify_many(tickets[:8], LABELS)))[:8]:
    print(f"{res.label:10s} {res.confidence:.2f}  {text[:70]}")

## 4. Background jobs

A job runs server-side and survives a dropped connection or a restart (it resumes). Submit, wait, read the
results page by page. Jobs of different API keys take turns fairly; inside one key the oldest job goes first and the
newer ones show `waiting_reason: "behind"`.

In [ ]:
job = clef.classify_job(tickets, labels=LABELS)
print("submitted", job.id, job.status)
job = clef.wait_job(job.id, timeout=600, on_progress=lambda j: print(f"  {j.status} {j.done}/{j.total}"))
print("finished:", job.status)
rows = list(clef.job_results(job.id))
print(len(rows), "rows; first:", rows[0].result.label if rows[0].ok else rows[0].error)

## 5. Evaluate on labelled data

`POST /v1/evaluate` returns accuracy, per-label metrics, calibration and per-row predictions. The 47 bundled
tickets include a few deliberately ambiguous ones.

In [ ]:
import httpx

H = {"X-API-Key": os.environ["CLEF_API_KEY"]} if os.environ.get("CLEF_API_KEY") else {}
rows = [{"input": r["ticket"], "gold": r["department"]} for r in csv.DictReader(open(repo / "examples" / "tickets_labelled.csv"))]
ev = httpx.post(os.environ["CLEF_URL"] + "/v1/evaluate", headers=H, timeout=600,
                json={"labels": list(LABELS), "rows": rows, "include_predictions": True}).json()
cal = ev.get("calibration") or {}
print("accuracy", round(ev["accuracy"], 3), "| macro F1", round(ev["macro_f1"], 3), "| ECE", cal.get("ece"))
wrong = [p for p in ev["predictions"] if not p["correct"]]
for p in wrong[:5]:
    print(f"  gold {p['gold']:9s} predicted {p['predicted']:9s} ({p['confidence']:.2f})  {p['input'][:60]}")

## 6. The OpenAI SDK and the Batch API

clef speaks enough of the OpenAI API for the official SDK: upload a JSONL file, create a batch, download the output.
Each line is one `/v1/classify` (or `/v1/chat/completions`, `/v1/score`) request.

In [ ]:
import io, json, time
from openai import OpenAI

oa = OpenAI(base_url=os.environ["CLEF_URL"] + "/v1", api_key=os.environ.get("CLEF_API_KEY", "none"))
lines = "\n".join(json.dumps({"custom_id": f"t{i}", "method": "POST", "url": "/v1/classify",
                               "body": {"input": t, "labels": list(LABELS)}}) for i, t in enumerate(tickets[:5]))
f = oa.files.create(file=("tickets.jsonl", io.BytesIO(lines.encode())), purpose="batch")
batch = oa.batches.create(input_file_id=f.id, endpoint="/v1/classify", completion_window="24h")
while batch.status not in ("completed", "failed", "cancelled", "expired"):
    time.sleep(1)
    batch = oa.batches.retrieve(batch.id)
print(batch.status, batch.request_counts)
for line in oa.files.content(batch.output_file_id).text.splitlines():
    d = json.loads(line)
    print(d["custom_id"], d["response"]["body"]["label"])

## 7. The self-evolving router and the Review queue

A router grows a classifier's label set. Texts it cannot place (`other`, low confidence) are buffered; every 20 of
them go to an LLM that proposes new labels, and clef re-classifies them to confirm a proposal before you see it in
**Review** (console) or via the API. It needs an LLM key on the server (`GEMINI_API_KEY=...` in `.env`, or any
OpenAI-compatible server via `CLEF_LLM_BASE_URL`): input text is sent to that host. Routing itself works without one.

In [ ]:
clef.put_router("notebook-router", seeds={"billing": "money, invoices, refunds", "technical": "bugs and outages"},
                domain_hint="customer support tickets", mode="review")
reasons = {}
for t in tickets:
    res = clef.route("notebook-router", t)
    reasons[res["reason"]] = reasons.get(res["reason"], 0) + 1
print("routed:", reasons)  # 'other' / 'low_confidence' items are buffered for the LLM

In [ ]:
# Ask for proposals now instead of waiting for 20 buffered items (needs the LLM key on the server).
try:
    print(clef.propose("notebook-router"))
except Exception as exc:  # 503: LLM off; 409: a call is already running for this router
    print("propose:", exc)

proposals = clef.list_review(router="notebook-router", kind="proposal")["items"]
if not proposals:
    print("no proposal passed clef's re-check this time (route more texts, or try propose again later)")
for item in proposals:
    for lab in item["proposal"]["labels"]:
        print(item["id"], "->", lab["name"], ":", lab["description"], "| re-check passed:", lab["acceptance"]["passed"])

In [ ]:
# Approve the first proposal: it is published as a new version of the router's classifier.
items = clef.list_review(router="notebook-router", kind="proposal", state="open")["items"]
if items:
    clef.resolve_review(items[0]["id"], "approve")
    print("labels now:", list(clef.get_router("notebook-router")["labels"]))
else:
    print("nothing to approve yet")

## 8. API keys (admin)

With auth on (`CLEF_API_KEY` or `CLEF_API_KEYS` on the server), an admin key can create scoped keys with quotas.
Scopes: `classify`, `jobs`, `classifiers`, `ops`, `admin`, `urgent`. The plaintext key is shown once.
The console's **Keys** tab does the same with usage sparklines.

In [ ]:
if H:
    k = httpx.post(os.environ["CLEF_URL"] + "/v1/admin/keys", headers=H,
                   json={"name": "notebook-reader", "scopes": ["classify"], "daily_quota": 1000}).json()
    print("new key (shown once):", k.get("key", k))
    print([x["name"] for x in httpx.get(os.environ["CLEF_URL"] + "/v1/admin/keys", headers=H).json()["principals"]])
else:
    print("auth is off on this server: no keys to manage")

## 9. Performance settings (ROCm / RX 7900 XTX)

| Variable | Effect (measured on the 7900 XTX) |
| --- | --- |
| `CLEF_TUNABLEOP=1` | picks the fastest GEMM kernel per shape for this GPU: +27-32 % req/s on mixed-length traffic, single requests p50 105 vs 142 ms. The server is ready in seconds; it tunes in the background while idle (~1 h of idle time once per GPU / driver / ROCm / torch version; keep the state dir). Progress below. |
| `CLEF_MASK_FREE=1`, `CLEF_FAST_HEAD=1` | another +5-12 % on top of TunableOp |
| `CLEF_JOB_SORT_WINDOW=512` | +14 % job throughput but a worse interactive p95; off by default |

Probabilities move by up to ~0.03 with these on (different GEMM summation order); top answers only change on
near coin-flip questions. `clef tune` (server stopped) tunes the GEMMs live traffic recorded as untuned.

In [ ]:
stats = clef.stats()
print("status:", stats["status"], "| TunableOp background tuning:", stats.get("tunableop"))

## Clean up

In [ ]:
triage.delete()
clef.delete_router("notebook-router")
clef.close() if hasattr(clef, "close") else None